In [3]:
!pip install -U qwen-asr jiwer pandas soundfile

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 45.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 64.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import torch
import subprocess

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")

# Show NVIDIA driver/GPU information
!nvidia-smi

PyTorch version: 2.14.1+cu130
CUDA available: True
PyTorch CUDA version: 13.0
GPU: Tesla T4
GPU memory (GB): 14.56
Fri Nov 27 21:56:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                        

In [4]:
import torch
import time
from qwen_asr import Qwen3ASRModel

assert torch.cuda.is_available(), "CUDA GPU is required!"

MODEL_NAME = "Qwen/Qwen3-ASR-1.7B"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

load_start = time.perf_counter()

model = Qwen3ASRModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="cuda:0",
    max_inference_batch_size=1,
    max_new_tokens=256,
)

torch.cuda.synchronize()
load_time = time.perf_counter() - load_start

print(f"Model loaded in {load_time:.2f} seconds")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.22G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

... (5 outputs left out)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Model loaded in 80.86 seconds
GPU memory allocated: 3.80 GB
GPU memory reserved: 4.41 GB


In [5]:
!pip install -q datasets

In [14]:
from datasets import load_dataset
import soundfile as sf
import os

os.makedirs("samples", exist_ok=True)

# Load test splits
arabic = load_dataset(
    "google/fleurs", "ar_eg", split="test"
)

english = load_dataset(
    "google/fleurs", "en_us", split="test"
)

# Select three samples from each
selected = [
    ("ar", arabic[i]) for i in range(10,13)
] + [
    ("en", english[i]) for i in range(10,13)
]

ground_truth = []

for i, (language, sample) in enumerate(selected):

    audio = sample["audio"]

    filename = f"samples/{language}_{i}.wav"

    sf.write(
        filename,
        audio["array"],
        audio["sampling_rate"]
    )

    ground_truth.append({
        "audio": filename,
        "language": language,
        "reference": sample["transcription"]
    })

for item in ground_truth:
    print(item)

{'audio': 'samples/ar_0.wav', 'language': 'ar', 'reference': 'في حال كنت تعبر منطقة بحر البلطيق الشمالية في الشتاء فيجب أن تفحص موقع الكابينة إذ أن المرور عبر الجليد يسبب ضوضاء صاخبة للغاية للأشخاص الأكثر تضرراً'}
{'audio': 'samples/ar_1.wav', 'language': 'ar', 'reference': 'أُرسل القمر الصناعي إلى الفضاء بواسطة صاروخ يستخدم العلماء التلسكوبات في الفضاء لأن الغلاف الجوي لكوكب الأرض يشوه بعض الضوء والرؤية'}
{'audio': 'samples/ar_2.wav', 'language': 'ar', 'reference': 'بصفة عامة، يمكن أن يظهر سلوكان عندما يبدأ المديرون في قيادة أقرانهم السابقين أحدهما هو محاولة البقاء مثل الآخرين أو الأخريات'}
{'audio': 'samples/en_3.wav', 'language': 'en', 'reference': 'through the night between 150 and 200 copies were made now known as dunlap broadsides'}
{'audio': 'samples/en_4.wav', 'language': 'en', 'reference': 'for example each year students from bennet school in north carolina design a website about their trip to the state capital each year the website gets remodeled but old versions are kept onl

In [15]:
from google.colab import files
import glob
# uploaded = files.upload()
audio_files = list(glob.glob('samples/*.wav'))
# audio_files = list(uploaded.keys())

# print("Uploaded audio files:")
# for path in audio_files:
#     print("-", path)

In [16]:

import soundfile as sf
import pandas as pd
import time
import torch
from jiwer import wer, cer

results = []

# Normalize language codes to the names expected by Qwen3-ASR
language_names = {
    "ar": "Arabic",
    "en": "English"
}

# Warm-up (excluded from measurements)
if ground_truth:
    _ = model.transcribe(
        audio=ground_truth[0]["audio"],
        language=None
    )
    torch.cuda.synchronize()


def run_inference(audio_path, language):
    """Run inference and return prediction and execution time."""

    torch.cuda.synchronize()
    start = time.perf_counter()

    output = model.transcribe(
        audio=audio_path,
        language=language
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    return output[0], elapsed


for sample in ground_truth:

    audio_path = sample["audio"]
    reference = sample["reference"]
    language_code = sample["language"]
    specified_language = language_names[language_code]

    duration = sf.info(audio_path).duration

    # ==========================================
    # 1. Automatic language detection
    # ==========================================

    auto_output, auto_time = run_inference(
        audio_path,
        language=None
    )

    auto_prediction = auto_output.text

    auto_wer = wer(reference, auto_prediction)
    auto_cer = cer(reference, auto_prediction)

    auto_rtf = auto_time / duration

    # ==========================================
    # 2. Language explicitly specified
    # ==========================================

    given_output, given_time = run_inference(
        audio_path,
        language=specified_language
    )

    given_prediction = given_output.text

    given_wer = wer(reference, given_prediction)
    given_cer = cer(reference, given_prediction)

    given_rtf = given_time / duration

    # ==========================================
    # Store results
    # ==========================================

    results.append({
        "Audio": audio_path,
        "Reference Language": specified_language,
        "Duration (s)": duration,
        "Reference": reference,

        "Auto Detected Language": auto_output.language,
        "Auto Prediction": auto_prediction,
        "Auto WER": auto_wer,
        "Auto CER": auto_cer,
        "Auto Time (s)": auto_time,
        "Auto RTF": auto_rtf,

        "Specified Prediction": given_prediction,
        "Specified WER": given_wer,
        "Specified CER": given_cer,
        "Specified Time (s)": given_time,
        "Specified RTF": given_rtf
    })

    print("\n" + "=" * 80)
    print("Audio:", audio_path)
    print("Reference language:", specified_language)
    print("Reference:", reference)

    print("\n--- Automatic Detection ---")
    print("Detected language:", auto_output.language)
    print("Prediction:", auto_prediction)
    print(f"WER: {auto_wer:.4f} | CER: {auto_cer:.4f}")
    print(f"Time: {auto_time:.2f}s | RTF: {auto_rtf:.3f}")

    print("\n--- Specified Language ---")
    print("Language:", specified_language)
    print("Prediction:", given_prediction)
    print(f"WER: {given_wer:.4f} | CER: {given_cer:.4f}")
    print(f"Time: {given_time:.2f}s | RTF: {given_rtf:.3f}")


# ==========================================
# Results and averages
# ==========================================

df = pd.DataFrame(results)

metric_columns = [
    "Auto WER",
    "Auto CER",
    "Auto Time (s)",
    "Auto RTF",
    "Specified WER",
    "Specified CER",
    "Specified Time (s)",
    "Specified RTF"
]

averages = df[metric_columns].mean()

print("\n" + "=" * 80)
print("EXPERIMENT SUMMARY")
print("=" * 80)

print("\nAverage Metrics:")
display(averages.to_frame(name="Average").round(4))

print("\nPer-Sample Results:")
display(df)

# Save detailed results
df.to_csv("qwen3_asr_results.csv", index=False)

# Save summary separately
averages.to_csv("qwen3_asr_average_results.csv")

print("\nSaved:")
print("- qwen3_asr_results.csv")
print("- qwen3_asr_average_results.csv")


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



Audio: samples/ar_0.wav
Reference language: Arabic
Reference: في حال كنت تعبر منطقة بحر البلطيق الشمالية في الشتاء فيجب أن تفحص موقع الكابينة إذ أن المرور عبر الجليد يسبب ضوضاء صاخبة للغاية للأشخاص الأكثر تضرراً

--- Automatic Detection ---
Detected language: Arabic
Prediction: في حال كنت تعبر منطقة بحر البلطيق الشمالية في الشتاء، فيجب أن تفحص موقع الكابينة إذ أن المرور عبر الجليد يسبب ضوضاء شخبة للغاية للأشخاص الأقصر تضرراً.
WER: 0.1481 | CER: 0.0403
Time: 2.55s | RTF: 0.217

--- Specified Language ---
Language: Arabic
Prediction: في حال كنت تعبر منطقة بحر البلطيق الشمالية في الشتاء، فيجب أن تفحص موقع الكابينة إذ أن المرور عبر الجليد يسبب ضوضاء شخبة للغاية للأشخاص الأقصر تضرراً.
WER: 0.1481 | CER: 0.0403
Time: 3.12s | RTF: 0.265


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



Audio: samples/ar_1.wav
Reference language: Arabic
Reference: أُرسل القمر الصناعي إلى الفضاء بواسطة صاروخ يستخدم العلماء التلسكوبات في الفضاء لأن الغلاف الجوي لكوكب الأرض يشوه بعض الضوء والرؤية

--- Automatic Detection ---
Detected language: Arabic
Prediction: أرسل القمر الصناعي إلى الفضاء بواسطة صاروخ. يستخدم العلماء التلسكوبات في الفضاء لأن الغلاف الجوي لكوكب الأرض يشوه بعض الضوء والرؤية.
WER: 0.1429 | CER: 0.0229
Time: 2.25s | RTF: 0.197

--- Specified Language ---
Language: Arabic
Prediction: أرسل القمر الصناعي إلى الفضاء بواسطة صاروخ. يستخدم العلماء التلسكوبات في الفضاء لأن الغلاف الجوي لكوكب الأرض يشوه بعض الضوء والرؤية.
WER: 0.1429 | CER: 0.0229
Time: 2.13s | RTF: 0.187


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



Audio: samples/ar_2.wav
Reference language: Arabic
Reference: بصفة عامة، يمكن أن يظهر سلوكان عندما يبدأ المديرون في قيادة أقرانهم السابقين أحدهما هو محاولة البقاء مثل الآخرين أو الأخريات

--- Automatic Detection ---
Detected language: Arabic
Prediction: بصفة عامة يمكن أن يظهر سلوكيات عندما يبدأ المديرون في قيادة أقرانهم السابقين، أحدهما هو محاولة البقاء مثل الآخرين أو الأخريات.
WER: 0.1905 | CER: 0.0403
Time: 2.32s | RTF: 0.190

--- Specified Language ---
Language: Arabic
Prediction: بصفة عامة يمكن أن يظهر سلوكيات عندما يبدأ المديرون في قيادة أقرانهم السابقين، أحدهما هو محاولة البقاء مثل الآخرين أو الأخريات.
WER: 0.1905 | CER: 0.0403
Time: 2.19s | RTF: 0.179


... (6 outputs left out)

,Average
Auto WER,0.1973
Auto CER,0.0487
Auto Time (s),2.1261
Auto RTF,0.2155
Specified WER,0.1973
Specified CER,0.0487
Specified Time (s),2.1119
Specified RTF,0.2147



Per-Sample Results:


,Audio,Reference Language,Duration (s),Reference,Auto Detected Language,Auto Prediction,Auto WER,Auto CER,Auto Time (s),Auto RTF,Specified Prediction,Specified WER,Specified CER,Specified Time (s),Specified RTF
0,samples/ar_0.wav,Arabic,11.76,في حال كنت تعبر منطقة بحر البلطيق الشمالية في ...,Arabic,في حال كنت تعبر منطقة بحر البلطيق الشمالية في ...,0.148148,0.040268,2.549926,0.216830,في حال كنت تعبر منطقة بحر البلطيق الشمالية في ...,0.148148,0.040268,3.116832,0.265037
1,samples/ar_1.wav,Arabic,11.40,أُرسل القمر الصناعي إلى الفضاء بواسطة صاروخ يس...,Arabic,أرسل القمر الصناعي إلى الفضاء بواسطة صاروخ. يس...,0.142857,0.022901,2.248361,0.197225,أرسل القمر الصناعي إلى الفضاء بواسطة صاروخ. يس...,0.142857,0.022901,2.130827,0.186915
2,samples/ar_2.wav,Arabic,12.24,بصفة عامة، يمكن أن يظهر سلوكان عندما يبدأ المد...,Arabic,بصفة عامة يمكن أن يظهر سلوكيات عندما يبدأ المد...,0.190476,0.040323,2.321214,0.189642,بصفة عامة يمكن أن يظهر سلوكيات عندما يبدأ المد...,0.190476,0.040323,2.193254,0.179187
3,samples/en_3.wav,English,4.80,through the night between 150 and 200 copies w...,English,For the night between 150 and 200 copies were ...,0.266667,0.105882,1.542029,0.321256,For the night between 150 and 200 copies were ...,0.266667,0.105882,1.575079,0.328142
4,samples/en_4.wav,English,11.28,for example each year students from bennet sch...,English,"For example, each year students from Bennett S...",0.263158,0.050926,2.309659,0.204757,"For example, each year students from Bennett S...",0.263158,0.050926,2.018141,0.178913
5,samples/en_5.wav,English,10.92,lion prides act much like packs of wolves or d...,English,Lion prides act much like packs of wolves or d...,0.172414,0.031847,1.785641,0.163520,Lion prides act much like packs of wolves or d...,0.172414,0.031847,1.637019,0.149910



Saved:
- qwen3_asr_results.csv
- qwen3_asr_average_results.csv


In [9]:
from google.colab import files

files.download("qwen3_asr_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>